In [ ]:
# =============================================================================
# XGBoost Wildfire Ignition Model — Ontario
# Study period: 2010–2024
# Train: 2010-2020  |  Validation: 2021–2022  |  Test: 2023–2024 (locked)
# Features: 15 (10 dynamic + 5 static)
# =============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, balanced_accuracy_score)
import glob
import os
import joblib
import time

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'
os.makedirs('models', exist_ok=True)

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES = DYNAMIC_FEATURES + STATIC_FEATURES   # 15 features total

SEASON_BUFFER_DAYS = 30

TRAIN_YEARS = list(range(2010, 2021))   # 2010–2020
VAL_YEARS   = [2021, 2022]              # 2021–2022
# TEST_YEARS = [2023, 2024]             # locked — not loaded here

# ── 1. Load CSVs ──────────────────────────────────────────────────────────────
def load_years(years, split_label):
    """Load all monthly CSVs for a list of years and return concatenated DataFrame."""
    all_dfs = []
    missing = []

    for year in years:
        pattern = os.path.join(DATA_DIR, f'ML_{split_label}_{year}*.csv')
        files   = sorted(glob.glob(pattern))

        if not files:
            missing.append(year)
            continue

        year_dfs = []
        for f in files:
            try:
                df = pd.read_csv(f)
                year_dfs.append(df)
            except Exception as e:
                print(f"  WARNING: could not read {os.path.basename(f)}: {e}")

        if year_dfs:
            year_df = pd.concat(year_dfs, ignore_index=True)
            n_fire  = (year_df['target_ignition'] == 1).sum() \
                      if 'target_ignition' in year_df.columns else 0
            print(f"  {year}: {len(year_df):>12,} rows  ({n_fire:,} fire pixels)")
            all_dfs.append(year_df)

    if missing:
        print(f"  WARNING: no CSV files found for years: {missing}")
        print(f"  Make sure ML_{split_label}_YYYY*.csv files are in {DATA_DIR}")

    if not all_dfs:
        raise FileNotFoundError(
            f"No CSV files found for {split_label} years {years} in {DATA_DIR}")

    return pd.concat(all_dfs, ignore_index=True)


print("Loading training years (2010–2020)...")
df_train_raw = load_years(TRAIN_YEARS, 'train')
print(f"  Total: {len(df_train_raw):,} rows\n")

print("Loading validation years (2021–2022)...")
df_val_raw = load_years(VAL_YEARS, 'val')   
print(f"  Total: {len(df_val_raw):,} rows")

# ── 2. Data cleaning ──────────────────────────────────────────────────────────
def clean_df(df, name):
    required = FEATURES + ['target_ignition', 'fire_cause', 'date']
    n_before = len(df)

    df = df.dropna(subset=required)
    df['target_ignition'] = df['target_ignition'].astype(int)
    df['fire_cause']      = df['fire_cause'].astype(int)
    df['land_cover']      = df['land_cover'].astype(int)
    df = df[df['target_ignition'].isin([0, 1])]
    df = df[df['fire_cause'].between(0, 5)]

    print(f"  {name}: {n_before:,} → {len(df):,} rows after cleaning "
          f"({n_before - len(df):,} dropped)")
    return df

print("\nCleaning data...")
df_train_raw = clean_df(df_train_raw, 'train')
df_val_raw   = clean_df(df_val_raw,   'val')


# ── Check suspicious early and late season fires ──────────────────────────────

df_train_raw['date_parsed'] = pd.to_datetime(df_train_raw['date'], format='%Y%m%d')
df_train_raw['doy']         = df_train_raw['date_parsed'].dt.dayofyear

winter_fires = df_train_raw[
    (df_train_raw['target_ignition'] == 1) &
    ((df_train_raw['doy'] < 91) | (df_train_raw['doy'] > 310))
][['date', 'longitude', 'latitude', 'doy', 'temp_max']].sort_values('doy')

print(f"Out-of-season fire pixels (DOY < 91 or DOY > 310):")
print(f"Total: {len(winter_fires)} records\n")
print(winter_fires.to_string())

# Clean up temporary columns before filtering
df_train_raw = df_train_raw.drop(columns=['date_parsed', 'doy'])



In [ ]:

DOY_START_HARD = 91    # April 1
DOY_END_HARD   = 310   # November 6

def apply_hard_season_filter(df, name):
    df = df.copy()
    df['date_parsed'] = pd.to_datetime(df['date'], format='%Y%m%d')
    df['doy']         = df['date_parsed'].dt.dayofyear

    n_before   = len(df)
    n_fire_in  = (df['target_ignition'] == 1).sum()

    df = df[(df['doy'] >= DOY_START_HARD) &
            (df['doy'] <= DOY_END_HARD)].copy()

    n_after    = len(df)
    n_fire_out = (df['target_ignition'] == 1).sum()
    n_removed  = n_fire_in - n_fire_out

    print(f"  {name}: {n_before:,} → {n_after:,} rows  "
          f"({(1-n_after/n_before)*100:.1f}% removed)")
    print(f"  Fires: {n_fire_in:,} → {n_fire_out:,}  "
          f"({n_removed} out-of-season records excluded)")

    return df.drop(columns=['date_parsed', 'doy'])

print("Applying hard calendar filter (DOY 91–310)...")
df_train = apply_hard_season_filter(df_train_raw, 'train')
df_val   = apply_hard_season_filter(df_val_raw,   'val')

# ── Load CV-optimised threshold ───────────────────────────────────────────────

threshold_record  = joblib.load('models/xgb_cv_threshold.pkl')
CV_THRESHOLD      = threshold_record['cv_threshold']

print(f"Loaded CV threshold: {CV_THRESHOLD:.4f}")

# ── 4. Class balance ──────────────────────────────────────────────────────────
def print_balance(df, name):
    n_pos = (df['target_ignition'] == 1).sum()
    n_neg = (df['target_ignition'] == 0).sum()
    ratio = n_neg / max(n_pos, 1)
    print(f"  {name}: {n_pos:,} fires  {n_neg:,} no-fire  "
          f"ratio {ratio:.0f}:1")
    return ratio

print("\nClass balance:")
ratio_tr = print_balance(df_train, 'train')
ratio_vl = print_balance(df_val,   'val')

if (df_train['target_ignition'] == 1).sum() == 0:
    raise ValueError("No fire pixels in training set — cannot train.")

# ── 5. Prepare arrays ─────────────────────────────────────────────────────────
# Sort by date to preserve temporal order
df_train = df_train.sort_values('date').reset_index(drop=True)
df_val   = df_val.sort_values('date').reset_index(drop=True)

X_train     = df_train[FEATURES].values
y_ign_train = df_train['target_ignition'].values
X_val       = df_val[FEATURES].values
y_ign_val   = df_val['target_ignition'].values

fire_tr_mask  = df_train['target_ignition'] == 1
fire_vl_mask  = df_val['target_ignition']   == 1
X_cause_train = df_train.loc[fire_tr_mask, FEATURES].values
y_cause_train = df_train.loc[fire_tr_mask, 'fire_cause'].values
X_cause_val   = df_val.loc[fire_vl_mask,   FEATURES].values
y_cause_val   = df_val.loc[fire_vl_mask,   'fire_cause'].values

print(f"\nArray shapes:")
print(f"  X_train       : {X_train.shape}")
print(f"  X_val         : {X_val.shape}")
print(f"  X_cause_train : {X_cause_train.shape}  (fire pixels only)")
print(f"  X_cause_val   : {X_cause_val.shape}")

# ── 6. Model: Binary ignition ───────────────────────────────────────────────
print("\n" + "="*60)
print(f"Model A: target_ignition  "
      f"(train {TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]}, "
      f"val {VAL_YEARS[0]}–{VAL_YEARS[-1]})")
print("="*60)

t0 = time.time()

model_ignition = xgb.XGBClassifier(
    objective             = 'binary:logistic',
    n_estimators          = 500,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 10,
    scale_pos_weight      = ratio_tr,
    eval_metric           = 'auc',
    early_stopping_rounds = 20,
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 1
)

model_ignition.fit(
    X_train, y_ign_train,
    eval_set = [(X_val, y_ign_val)],
    verbose  = 50
)

t1 = time.time()
print(f"\nTraining time : {(t1-t0)/60:.1f} minutes")
print(f"Best iteration: {model_ignition.best_iteration}")

y_pred_prob = model_ignition.predict_proba(X_val)[:, 1]
y_pred      = (y_pred_prob >= CV_THRESHOLD).astype(int)  # ← CV threshold

print(f"\nValidation metrics ({VAL_YEARS[0]}–{VAL_YEARS[-1]})  "
      f"[threshold = {CV_THRESHOLD:.4f}]:")
print(f"  ROC-AUC           : {roc_auc_score(y_ign_val, y_pred_prob):.4f}")
print(f"  Balanced accuracy : {balanced_accuracy_score(y_ign_val, y_pred):.4f}")
print(f"\nClassification report:")
print(classification_report(y_ign_val, y_pred,
                            target_names=['No fire', 'Fire'],
                            digits=4, zero_division=0))
cm = confusion_matrix(y_ign_val, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Confusion matrix:")
print(f"  TN={tn:,}  FP={fp:,}")
print(f"  FN={fn:,}  TP={tp:,}")
print(f"  False positive rate : {fp/(fp+tn):.4f}  "
      f"({fp/(fp+tn)*100:.1f}% of no-fire days flagged)")
print(f"  Recall (fire)       : {tp/(tp+fn):.4f}  "
      f"({tp} of {tp+fn} fires detected)")

print(f"\nFeature importances (Model A):")
for feat, imp in sorted(zip(FEATURES, model_ignition.feature_importances_),
                         key=lambda x: -x[1]):
    print(f"  {feat:<22} {imp:.4f}")

# ── Save model with threshold metadata ────────────────────────────────────────
joblib.dump(model_ignition, 'models/model_ignition_2010_2022.pkl')
joblib.dump({'model'         : model_ignition,
             'cv_threshold'  : CV_THRESHOLD,
             'features'      : FEATURES,
             'train_years'   : TRAIN_YEARS,
             'val_years'     : VAL_YEARS},
            'models/model_ignition_2010_2022_with_threshold.pkl')

print(f"Saved: models/model_ignition_2010_2022_with_threshold.pkl")
print(f"  CV threshold stored: {CV_THRESHOLD:.4f}")


# ── Update summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("TRAINING SUMMARY — 2010–2022 (CV threshold applied)")
print(f"{'='*60}")
print(f"  Train years              : {TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]}")
print(f"  Val years                : {VAL_YEARS[0]}–{VAL_YEARS[-1]}")
print(f"  Train rows               : {len(df_train):,}")
print(f"  Val rows                 : {len(df_val):,}")
print(f"  Features                 : {len(FEATURES)}")
print(f"  Fire season              : DOY {DOY_START_HARD}–{DOY_END_HARD}")
print(f"  Classification threshold : {CV_THRESHOLD:.4f}  (CV-optimised)")
print(f"  ROC-AUC                  : {roc_auc_score(y_ign_val, y_pred_prob):.4f}")
print(f"  Recall (fire)            : {tp/(tp+fn):.4f}")
print(f"  False positive rate      : {fp/(fp+tn):.4f}")
print(f"\n  Note: model.predict() uses 0.5 by default.")
print(f"  Always use model.predict_proba()[:,1] >= {CV_THRESHOLD:.4f}")
print(f"  when applying this model to new data.")


In [ ]:
"""
XGBoost Wildfire Model — Interpretability Figures
  1. XGBoost native feature importance (gain)
  2. Permutation importance on validation set
  3. LIME explanations for individual predictions
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
import glob     

# Constants
PERM_NOFIRE_SAMPLE = 50000   
LIME_N_FIRE        = 5       
LIME_N_NOFIRE      = 3       

# ── 0. Load models and validation data ───────────────────────────────────────
print("Loading models and data...")

model_bundle    = joblib.load('models/model_ignition_2010_2022_with_threshold.pkl')
model_ignition  = model_bundle['model']        # XGBoost model object
CV_THRESHOLD    = model_bundle['cv_threshold'] # 0.7339
FEATURES        = model_bundle['features']     # list of 15 feature names

print(f"  Model loaded  : XGBoost ignition")
print(f"  CV threshold  : {CV_THRESHOLD:.4f}")
print(f"  Features      : {len(FEATURES)}")

cause_remap_inv = joblib.load('models/cause_remap_inv.pkl')
DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES = DYNAMIC_FEATURES + STATIC_FEATURES

# Human-readable feature labels for plots
FEATURE_LABELS = {
    'temp_max'        : 'Daily max temperature (°C)',
    'rh_min'          : 'Daily min relative humidity (%)',
    'vpd_mean'        : 'Mean VPD (kPa)',
    'wind_speed_mean' : 'Mean wind speed (m/s)',
    'precip_sum'      : 'Daily precipitation (mm)',
    'solar_rad_mj'    : 'Solar radiation (MJ/m²)',
    'rh_7d'           : '7-day mean RH (%)',
    'temp_7d'         : '7-day mean temperature (°C)',
    'precip_30d'      : '30-day precipitation (mm)',
    'recovery_value'  : 'Post-fire recovery [0–1]',
    'elevation_m'     : 'Elevation (m)',
    'slope_deg'       : 'Slope (degrees)',
    'land_cover'      : 'Land cover class',
    'pop_density'     : 'Population density',
    'road_proximity_m': 'Distance to road (m)',
}


os.makedirs('figures', exist_ok=True)

# ── Load a sample of validation data for permutation importance and LIME ──────
# Loading the full validation set would be too large for LIME.
# Sample a manageable subset: all fire pixels + random no-fire sample.
print("Loading validation sample...")

# ── Load validation sample — corrected for actual file naming ─────────────────
DATA_DIR  = r'C:\Users\lambe\RU_Thesis_2026\ml_data'
VAL_YEARS = [2021, 2022]   # actual validation years
VAL_LABEL = 'val'          # actual file label

DOY_START = 91
DOY_END   = 310

val_dfs = []
for year in VAL_YEARS:
    pattern = os.path.join(DATA_DIR, f'ML_{VAL_LABEL}_{year}*.csv')
    files   = sorted(glob.glob(pattern))
    if not files:
        print(f"WARNING: no files found for {year} matching {pattern}")
        continue
    for f in files:
        df = pd.read_csv(f)
        val_dfs.append(df)
    print(f"  {year}: {len(files)} files loaded")

if not val_dfs:
    raise FileNotFoundError(
        f"No validation files found in {DATA_DIR}.\n"
        f"Expected: ML_val_2021*.csv ... ML_val_2022*.csv")

df_val = pd.concat(val_dfs, ignore_index=True)
df_val['date_parsed'] = pd.to_datetime(df_val['date'], format='%Y%m%d')
df_val['doy']         = df_val['date_parsed'].dt.dayofyear
df_val = df_val[(df_val['doy'] >= DOY_START) & (df_val['doy'] <= DOY_END)]
df_val = df_val.dropna(subset=FEATURES + ['target_ignition', 'fire_cause'])
df_val['target_ignition'] = df_val['target_ignition'].astype(int)
df_val['fire_cause']      = df_val['fire_cause'].astype(int)

fire_val   = df_val[df_val['target_ignition'] == 1].reset_index(drop=True)
nofire_val = df_val[df_val['target_ignition'] == 0].sample(
    n=min(PERM_NOFIRE_SAMPLE, len(df_val[df_val['target_ignition'] == 0])),
    random_state=42).reset_index(drop=True)

print(f"\nValidation set loaded:")
print(f"  Fire pixels    : {len(fire_val):,}")
print(f"  No-fire sample : {len(nofire_val):,}")
print(f"  Date range     : {df_val['date'].min()} – {df_val['date'].max()}")

perm_df = pd.concat([fire_val, nofire_val], ignore_index=True)
X_perm  = perm_df[FEATURES].values
y_perm  = perm_df['target_ignition'].values

X_fire_cause = fire_val[FEATURES].values
y_fire_cause = fire_val['fire_cause'].values



# ── Colour palette ────────────────────────────────────────────────────────────
COL_IGNITION = '#c0392b'   # deep red — fire
COL_CAUSE    = '#2980b9'   # blue — cause model
COL_STATIC   = '#e67e22'   # orange — static features
COL_DYNAMIC  = '#27ae60'   # green — dynamic features
COL_GRID     = '#eeeeee'
FONT_TITLE   = 12
FONT_LABEL   = 10
FONT_TICK    = 9

def feature_colour(feat):
    return COL_DYNAMIC if feat in DYNAMIC_FEATURES else COL_STATIC

SOURCE_NOTE = ('Ontario wildfire ignition model  |  '
               'XGBoost  |  Training: 2010–2020  |  Validation: 2021–2022')

# =============================================================================
# Permutation importance — ignition model
# =============================================================================
print("Generating Figure 2: Permutation importance (ignition model)...")
print("This may take several minutes for large validation sets...")

from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

t0 = __import__('time').time()

pi_result = permutation_importance(
    model_ignition, X_perm, y_perm,
    n_repeats    = 10,
    scoring      = 'roc_auc',
    random_state = 42,
    n_jobs       = -1
)

t1 = __import__('time').time()
print(f"  Permutation importance computed in {(t1-t0)/60:.1f} minutes")

pi_means = pi_result.importances_mean
pi_stds  = pi_result.importances_std
idx      = np.argsort(pi_means)

feats_sorted  = [FEATURES[i] for i in idx]
labels_sorted = [FEATURE_LABELS[f] for f in feats_sorted]
colors_sorted = [feature_colour(f) for f in feats_sorted]
means_sorted  = pi_means[idx]
stds_sorted   = pi_stds[idx]

fig2, ax2 = plt.subplots(figsize=(10, 8))

bars = ax2.barh(range(len(means_sorted)), means_sorted,
                xerr=stds_sorted,
                color=colors_sorted,
                edgecolor='white', linewidth=0.5,
                height=0.7, zorder=3,
                error_kw={'elinewidth': 1.2, 'capsize': 3,
                          'ecolor': '#666666', 'capthick': 1.2})

# Highlight features where importance crosses zero (genuinely informative)
ax2.axvline(0, color='#333333', linewidth=1.0, linestyle='--', zorder=4)

for bar, mean, std in zip(bars, means_sorted, stds_sorted):
    if mean > 0:
        ax2.text(mean + std + 0.0003, bar.get_y() + bar.get_height()/2,
                 f'{mean:.4f}', va='center', ha='left',
                 fontsize=7.5, color='#333333')

ax2.set_yticks(range(len(labels_sorted)))
ax2.set_yticklabels(labels_sorted, fontsize=FONT_TICK)
ax2.set_xlabel('Decrease in ROC-AUC when feature is permuted\n'
               '(mean ± std over 10 repeats)',
               fontsize=FONT_LABEL)
ax2.set_title('Permutation Feature Importance — Ignition Model\n'
              'Validation set: 2021-2022  |  Metric: ROC-AUC',
              fontsize=FONT_TITLE, fontweight='bold', pad=8)
ax2.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
ax2.set_axisbelow(True)
ax2.spines[['top', 'right']].set_visible(False)

dyn_patch = mpatches.Patch(facecolor=COL_DYNAMIC, label='Dynamic (meteorological)')
sta_patch = mpatches.Patch(facecolor=COL_STATIC,  label='Static (terrain/socioeconomic)')
ax2.legend(handles=[dyn_patch, sta_patch], fontsize=8,
           loc='lower right', framealpha=0.9, edgecolor='#cccccc')

fig2.text(0.5, -0.01, SOURCE_NOTE,
          ha='center', fontsize=7.5, color='#888888', style='italic')

plt.tight_layout()
fig2.savefig('figures/fig_permutation_importance_ignition.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("  Saved: figures/fig_permutation_importance_ignition.png")
plt.close(fig2)


# =============================================================================
# LIME explanations — ignition model
# =============================================================================
print("\nGenerating Figure 4: LIME explanations (ignition model)...")
print("  Installing lime if needed...")

try:
    import lime
    import lime.lime_tabular
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime'])
    import lime
    import lime.lime_tabular

# Build LIME explainer on training data distribution
# We use a sample of the permutation set as the background distribution
print("  Building LIME explainer...")

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data   = X_perm,
    feature_names   = [FEATURE_LABELS[f] for f in FEATURES],
    class_names     = ['No fire', 'Fire'],
    mode            = 'classification',
    discretize_continuous = True,
    random_state    = 42
)

# ── Select LIME instances using CV threshold instead of 0.5 ──────────────────
# No-fire pixels closest to the CV threshold are the most uncertain
# under the actual operating conditions of the model

fire_probs   = model_ignition.predict_proba(fire_val[FEATURES].values)[:, 1]
nofire_probs = model_ignition.predict_proba(nofire_val[FEATURES].values)[:, 1]

top_fire_idx = np.argsort(fire_probs)[-LIME_N_FIRE:][::-1]
boundary_idx = np.argsort(np.abs(nofire_probs - CV_THRESHOLD))[:LIME_N_NOFIRE]
lime_instances = []
lime_labels    = []

for i in top_fire_idx:
    lime_instances.append(fire_val[FEATURES].values[i])
    lime_labels.append(
        f'Fire pixel  pred={fire_probs[i]:.2f}  '
        f'date={fire_val["date"].iloc[i]}')

for i in boundary_idx:
    lime_instances.append(nofire_val[FEATURES].values[i])
    lime_labels.append(
        f'No-fire pixel  pred={nofire_probs[i]:.2f}  '
        f'date={nofire_val["date"].iloc[i]}')

print(f"  Explaining {len(lime_instances)} instances...")

# Generate one subplot per instance
n_instances = len(lime_instances)
fig4, axes  = plt.subplots(n_instances, 1,
                            figsize=(12, 4.5 * n_instances))
if n_instances == 1:
    axes = [axes]

for ax, instance, label in zip(axes, lime_instances, lime_labels):
    exp = lime_explainer.explain_instance(
        data_row       = instance,
        predict_fn     = model_ignition.predict_proba,
        num_features   = 10,
        num_samples    = 1000
    )

    # Extract weights
    exp_list = exp.as_list()
    feat_names = [e[0] for e in exp_list]
    weights    = [e[1] for e in exp_list]

    # Sort by absolute weight
    order  = np.argsort(np.abs(weights))
    f_sorted = [feat_names[i] for i in order]
    w_sorted = [weights[i]    for i in order]

    colors = [COL_IGNITION if w > 0 else '#2980b9' for w in w_sorted]

    ax.barh(range(len(w_sorted)), w_sorted, color=colors,
            edgecolor='white', linewidth=0.5, height=0.7, zorder=3)
    ax.axvline(0, color='#333333', linewidth=0.8, zorder=4)
    ax.set_yticks(range(len(f_sorted)))
    ax.set_yticklabels(f_sorted, fontsize=8)
    ax.set_xlabel('LIME weight  (red = increases fire probability)',
                  fontsize=9)
    ax.set_title(label, fontsize=9, fontweight='bold', pad=5)
    ax.grid(axis='x', color=COL_GRID, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

    pos_patch = mpatches.Patch(facecolor=COL_IGNITION,
                               label='Increases fire probability')
    neg_patch = mpatches.Patch(facecolor='#2980b9',
                               label='Decreases fire probability')
    ax.legend(handles=[pos_patch, neg_patch], fontsize=7.5,
              loc='lower right', framealpha=0.9, edgecolor='#cccccc')

fig4.suptitle('LIME Local Explanations — Ignition Model\n'
              'Top fire predictions and uncertain no-fire pixels',
              fontsize=13, fontweight='bold', y=1.005)
fig4.text(0.5, -0.005, SOURCE_NOTE,
          ha='center', fontsize=7.5, color='#888888', style='italic')

plt.tight_layout()
fig4.savefig('figures/fig_lime_ignition.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("  Saved: figures/fig_lime_ignition.png")
plt.close(fig4)

# =============================================================================
# Combined summary — native vs permutation importance (ignition)
# =============================================================================
print("Generating Figure 6: Native vs permutation importance comparison...")

fig6, (ax6a, ax6b) = plt.subplots(1, 2, figsize=(16, 7))
fig6.subplots_adjust(wspace=0.05)

# Native importance — sorted by permutation importance for fair comparison
perm_order  = np.argsort(pi_means)
native_imp  = model_ignition.feature_importances_[perm_order]
perm_imp    = pi_means[perm_order]
perm_std    = pi_stds[perm_order]
feat_labels = [FEATURE_LABELS[FEATURES[i]] for i in perm_order]
colors_ord  = [feature_colour(FEATURES[i]) for i in perm_order]

ax6a.barh(range(len(native_imp)), native_imp,
          color=colors_ord, edgecolor='white', linewidth=0.5,
          height=0.7, zorder=3)
ax6a.set_yticks(range(len(feat_labels)))
ax6a.set_yticklabels(feat_labels, fontsize=FONT_TICK)
ax6a.set_xlabel('Importance (gain)', fontsize=FONT_LABEL)
ax6a.set_title('(a) XGBoost native importance\n(information gain)',
               fontsize=FONT_TITLE, fontweight='bold', pad=8)
ax6a.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
ax6a.set_axisbelow(True)
ax6a.spines[['top', 'right']].set_visible(False)

ax6b.barh(range(len(perm_imp)), perm_imp,
          xerr=perm_std,
          color=colors_ord, edgecolor='white', linewidth=0.5,
          height=0.7, zorder=3,
          error_kw={'elinewidth': 1.2, 'capsize': 3,
                    'ecolor': '#666666', 'capthick': 1.2})
ax6b.axvline(0, color='#333333', linewidth=1.0, linestyle='--', zorder=4)
ax6b.set_yticks(range(len(feat_labels)))
ax6b.set_yticklabels([], fontsize=FONT_TICK)   # shared y-axis labels
ax6b.set_xlabel('Decrease in ROC-AUC (mean ± std)', fontsize=FONT_LABEL)
ax6b.set_title('(b) Permutation importance\n(validation ROC-AUC decrease)',
               fontsize=FONT_TITLE, fontweight='bold', pad=8)
ax6b.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
ax6b.set_axisbelow(True)
ax6b.spines[['top', 'right']].set_visible(False)

dyn_patch = mpatches.Patch(facecolor=COL_DYNAMIC, label='Dynamic (meteorological)')
sta_patch = mpatches.Patch(facecolor=COL_STATIC,  label='Static (terrain/socioeconomic)')
ax6b.legend(handles=[dyn_patch, sta_patch], fontsize=8,
            loc='lower right', framealpha=0.9, edgecolor='#cccccc')

fig6.suptitle('Feature Importance Comparison — Ignition Model\n'
              'Native gain vs permutation importance on validation set',
              fontsize=13, fontweight='bold', y=1.01)
fig6.text(0.5, -0.01, SOURCE_NOTE,
          ha='center', fontsize=7.5, color='#888888', style='italic')

fig6.savefig('figures/fig_importance_comparison_ignition.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("  Saved: figures/fig_importance_comparison_ignition.png")
plt.close(fig6)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("All figures saved to figures/")
print("="*60)
print("  fig_permutation_importance_ignition.png — PI ignition model")
print("  fig_lime_ignition.png                  — LIME, ignition model")
print("  fig_importance_comparison_ignition.png  — native vs PI side-by-side")


In [ ]:
# =============================================================================
# Feature Correlation Analysis
# Identifies highly correlated feature pairs that may be redundant
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# ── Load a representative sample for correlation analysis ─────────────────────
# Using the permutation importance sample (fire + 50k no-fire from val years)
# which is already loaded — or reload here if running standalone

print("Computing feature correlations...")


df_corr = pd.DataFrame(X_perm, columns=FEATURES)
n_sample = len(df_corr)
print(f"  Sample size: {n_sample:,} pixel-day observations")

# ── Pearson correlation (linear relationships) ─────────────────────────────────
pearson_corr = df_corr.corr(method='pearson')

# ── Spearman correlation (monotonic relationships — more robust) ───────────────
spearman_corr = df_corr.corr(method='spearman')

# ── Identify highly correlated pairs ──────────────────────────────────────────
THRESHOLD = 0.7   # flag pairs with |r| > 0.7

print(f"\nHighly correlated feature pairs (|Spearman r| > {THRESHOLD}):")
print("-" * 70)

high_corr_pairs = []
for i in range(len(FEATURES)):
    for j in range(i + 1, len(FEATURES)):
        r = spearman_corr.iloc[i, j]
        if abs(r) >= THRESHOLD:
            high_corr_pairs.append((FEATURES[i], FEATURES[j], r))
            direction = 'positive' if r > 0 else 'negative'
            print(f"  {FEATURES[i]:<22} ↔  {FEATURES[j]:<22}  "
                  f"r = {r:+.3f}  ({direction})")

if not high_corr_pairs:
    print("  No pairs above threshold — features are reasonably independent.")

# ── Figure 1: Spearman correlation heatmap ────────────────────────────────────
print("\nGenerating Figure 1: Correlation heatmap...")

# Human-readable short labels for heatmap
SHORT_LABELS = {
    'temp_max'        : 'temp_max',
    'rh_min'          : 'rh_min',
    'vpd_mean'        : 'vpd_mean',
    'wind_speed_mean' : 'wind_spd',
    'precip_sum'      : 'precip_sum',
    'solar_rad_mj'    : 'solar_rad',
    'rh_7d'           : 'rh_7d',
    'temp_7d'         : 'temp_7d',
    'precip_30d'      : 'precip_30d',
    'recovery_value'  : 'recovery',
    'elevation_m'     : 'elevation',
    'slope_deg'       : 'slope',
    'land_cover'      : 'land_cov',
    'pop_density'     : 'pop_dens',
    'road_proximity_m': 'road_prox',
}

labels = [SHORT_LABELS[f] for f in FEATURES]

# Diverging colormap: blue = negative, white = zero, red = positive
cmap = sns.diverging_palette(220, 10, as_cmap=True)

fig1, ax1 = plt.subplots(figsize=(12, 10))

mask = np.triu(np.ones_like(spearman_corr, dtype=bool), k=1)   # upper triangle only

sns.heatmap(
    spearman_corr,
    ax          = ax1,
    mask        = mask,
    annot       = True,
    fmt         = '.2f',
    cmap        = cmap,
    vmin        = -1,
    vmax        = 1,
    center      = 0,
    linewidths  = 0.5,
    linecolor   = 'white',
    annot_kws   = {'size': 8},
    xticklabels = labels,
    yticklabels = labels,
    square      = True,
    cbar_kws    = {'label': 'Spearman correlation coefficient',
                   'shrink': 0.8}
)

ax1.set_title(
    'Feature Correlation Matrix (Spearman ρ)\n'
    'Ontario fire model — validation sample 2021–2022',
    fontsize=12, fontweight='bold', pad=12
)
ax1.tick_params(axis='x', rotation=45, labelsize=9)
ax1.tick_params(axis='y', rotation=0,  labelsize=9)

# Draw rectangles around feature type groups
# Dynamic features: indices 0–9, Static features: 10–14
n_dyn = len(DYNAMIC_FEATURES)
n_sta = len(STATIC_FEATURES)

for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)

# Add group labels on both axes
ax1.add_patch(plt.Rectangle((0, 0), n_dyn, n_dyn,
              fill=False, edgecolor='#27ae60',
              linewidth=2.5, transform=ax1.transData, zorder=5))
ax1.add_patch(plt.Rectangle((n_dyn, n_dyn), n_sta, n_sta,
              fill=False, edgecolor='#e67e22',
              linewidth=2.5, transform=ax1.transData, zorder=5))

ax1.text(n_dyn/2, -0.7, 'Dynamic features',
         ha='center', va='bottom', fontsize=9,
         color='#27ae60', fontweight='bold',
         transform=ax1.transData)
ax1.text(n_dyn + n_sta/2, -0.7, 'Static',
         ha='center', va='bottom', fontsize=9,
         color='#e67e22', fontweight='bold',
         transform=ax1.transData)

fig1.text(0.5, -0.02,
    'Lower triangle only shown. Green box = dynamic features. '
    'Orange box = static features.',
    ha='center', fontsize=8, color='#666666', style='italic')

plt.tight_layout()
fig1.savefig('figures/fig_correlation_heatmap.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("  Saved: figures/fig_correlation_heatmap.png")
plt.close(fig1)

# ── Figure 2: Clustered heatmap (dendrogram) ──────────────────────────────────
print("Generating Figure 2: Clustered correlation heatmap...")

fig2 = plt.figure(figsize=(12, 10))

g = sns.clustermap(
    spearman_corr,
    cmap        = cmap,
    vmin        = -1,
    vmax        = 1,
    center      = 0,
    annot       = True,
    fmt         = '.2f',
    annot_kws   = {'size': 7.5},
    linewidths  = 0.5,
    linecolor   = 'white',
    xticklabels = labels,
    yticklabels = labels,
    figsize     = (13, 11),
    cbar_kws    = {'label': 'Spearman ρ', 'shrink': 0.6},
    dendrogram_ratio = 0.12
)

g.ax_heatmap.tick_params(axis='x', rotation=45, labelsize=8.5)
g.ax_heatmap.tick_params(axis='y', rotation=0,  labelsize=8.5)
g.fig.suptitle(
    'Clustered Feature Correlation Matrix (Spearman ρ)\n'
    'Hierarchical clustering groups similar features together',
    fontsize=12, fontweight='bold', y=1.01
)
g.fig.text(0.5, -0.01,
    'Dendrogram shows hierarchical clustering of features by correlation distance. '
    'Features that cluster together are most similar.',
    ha='center', fontsize=8, color='#666666', style='italic')

g.fig.savefig('figures/fig_correlation_clustered.png',
              dpi=300, bbox_inches='tight', facecolor='white')
print("  Saved: figures/fig_correlation_clustered.png")
plt.close('all')

# ── Figure 3: High-correlation pair scatterplots ──────────────────────────────
if high_corr_pairs:
    print("Generating Figure 3: High-correlation pair scatterplots...")

    n_pairs = len(high_corr_pairs)
    ncols   = min(3, n_pairs)
    nrows   = int(np.ceil(n_pairs / ncols))

    fig3, axes = plt.subplots(nrows, ncols,
                              figsize=(5.5 * ncols, 5 * nrows))
    axes = np.array(axes).flatten() if n_pairs > 1 else [axes]

    # Use a subsample for scatter speed
    scatter_sample = df_corr.sample(n=min(5000, len(df_corr)),
                                    random_state=42)

    for ax, (feat_a, feat_b, r) in zip(axes, high_corr_pairs):
        ax.scatter(scatter_sample[feat_a], scatter_sample[feat_b],
                   alpha=0.25, s=8, color='#2980b9', rasterized=True)

        # Regression line
        m, b = np.polyfit(scatter_sample[feat_a],
                          scatter_sample[feat_b], 1)
        x_range = np.linspace(scatter_sample[feat_a].min(),
                               scatter_sample[feat_a].max(), 100)
        ax.plot(x_range, m * x_range + b,
                color='#e74c3c', linewidth=1.8, zorder=5)

        ax.set_xlabel(FEATURE_LABELS[feat_a], fontsize=9)
        ax.set_ylabel(FEATURE_LABELS[feat_b], fontsize=9)
        ax.set_title(f'Spearman ρ = {r:+.3f}',
                     fontsize=10, fontweight='bold', pad=6)
        ax.grid(color='#eeeeee', linewidth=0.6)
        ax.spines[['top', 'right']].set_visible(False)

    # Hide unused panels
    for ax in axes[n_pairs:]:
        ax.set_visible(False)

    fig3.suptitle(
        f'Highly Correlated Feature Pairs  (|Spearman ρ| > {THRESHOLD})\n'
        'Scatterplots — subsample of 5,000 pixel-day observations',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    fig3.savefig('figures/fig_correlation_scatterplots.png',
                 dpi=300, bbox_inches='tight', facecolor='white')
    print("  Saved: figures/fig_correlation_scatterplots.png")
    plt.close(fig3)
else:
    print("No high-correlation pairs — scatterplot figure skipped.")

# ── Text summary ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("CORRELATION SUMMARY")
print("="*60)
print(f"\nSpearman correlation matrix ({len(FEATURES)} × {len(FEATURES)} features)")
print(f"Sample: {n_sample:,} pixel-day observations (val years 2021–2022)\n")

print(f"Highest correlations (top 10 by |r|):")
all_pairs = []
for i in range(len(FEATURES)):
    for j in range(i + 1, len(FEATURES)):
        all_pairs.append((FEATURES[i], FEATURES[j],
                          spearman_corr.iloc[i, j]))

all_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for feat_a, feat_b, r in all_pairs[:10]:
    flag = ' ← HIGH' if abs(r) >= THRESHOLD else ''
    print(f"  {feat_a:<22} ↔ {feat_b:<22}  r = {r:+.3f}{flag}")

print(f"\nFigures saved to figures/:")
print(f"  fig_correlation_heatmap.png      — full correlation matrix")
print(f"  fig_correlation_clustered.png    — clustered dendrogram heatmap")
if high_corr_pairs:
    print(f"  fig_correlation_scatterplots.png — high-correlation pair plots")


In [8]:
fig1, ax1 = plt.subplots(figsize=(12, 11))   # slightly taller to give room

mask = np.triu(np.ones_like(spearman_corr, dtype=bool), k=1)

sns.heatmap(
    spearman_corr,
    ax          = ax1,
    mask        = mask,
    annot       = True,
    fmt         = '.2f',
    cmap        = cmap,
    vmin        = -1,
    vmax        = 1,
    center      = 0,
    linewidths  = 0.5,
    linecolor   = 'white',
    annot_kws   = {'size': 8},
    xticklabels = labels,
    yticklabels = labels,
    square      = True,
    cbar_kws    = {'label': 'Spearman correlation coefficient',
                   'shrink': 0.8}
)

# Increase pad so title sits well above the heatmap
ax1.set_title(
    'Feature Correlation Matrix (Spearman ρ)\n'
    'Ontario fire model — validation sample 2021–2022',
    fontsize=12, fontweight='bold',
    pad=12            # ← increased from 12 to 40
)
ax1.tick_params(axis='x', rotation=45, labelsize=9)
ax1.tick_params(axis='y', rotation=0,  labelsize=9)

n_dyn = len(DYNAMIC_FEATURES)
n_sta = len(STATIC_FEATURES)

for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)

# Draw group boxes — unchanged
ax1.add_patch(plt.Rectangle((0, 0), n_dyn, n_dyn,
              fill=False, edgecolor='#27ae60',
              linewidth=2.5, transform=ax1.transData, zorder=5))
ax1.add_patch(plt.Rectangle((n_dyn, n_dyn), n_sta, n_sta,
              fill=False, edgecolor='#e67e22',
              linewidth=2.5, transform=ax1.transData, zorder=5))

# ── Move group labels BELOW the x-axis tick labels ────────────────────────────
# Use figure-level text with transform=ax1.transAxes so positioning is
# relative to the axes width and is not affected by data coordinates.
# y=-0.18 places the label below the rotated x-tick labels.

ax1.text(
    (n_dyn / 2) / len(FEATURES),          # x centre of dynamic block (axes fraction)
    -0.18,                                  # below x-axis tick labels
    'Dynamic features',
    ha='center', va='top',
    fontsize=9, color='#27ae60', fontweight='bold',
    transform=ax1.transAxes
)
ax1.text(
    (n_dyn + n_sta / 2) / len(FEATURES),  # x centre of static block (axes fraction)
    -0.18,
    'Static features',
    ha='center', va='top',
    fontsize=9, color='#e67e22', fontweight='bold',
    transform=ax1.transAxes
)

fig1.text(0.5, -0.04,
    'Lower triangle only shown.  '
    'Green box = dynamic features.  Orange box = static features.',
    ha='center', fontsize=8, color='#666666', style='italic')

plt.tight_layout()
fig1.savefig('figures/fig_correlation_heatmap.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("  Saved: figures/fig_correlation_heatmap.png")
plt.close(fig1)

  Saved: figures/fig_correlation_heatmap.png
